# SQL Baseline vs Cypher Comparison

This notebook demonstrates the difference between a traditional SQL (relational) approach and a Cypher (graph) approach for answering the same analytical questions about Eindhoven cycling infrastructure.

We use SQLite (built into Python) to simulate a relational database. The same CSV files used for Neo4j import are loaded into SQL tables, and we run equivalent queries to show the structural differences.

## 1. Setup: Load CSVs into SQLite tables

In [ ]:
import sqlite3
from pathlib import Path

import pandas as pd

# Create in-memory SQLite database
conn = sqlite3.connect(":memory:")

# Path relative to notebooks/sql_baseline_comparison.ipynb
DATA_DIR = Path("../data/sql")
RESULT_DIR = DATA_DIR / "results"
RESULT_DIR.mkdir(exist_ok=True)

# Load CSVs
neighbourhoods = pd.read_csv(DATA_DIR / "neighbourhood_summary.csv")
accidents = pd.read_csv(DATA_DIR / "accident_neighbourhood_join.csv")
lanes = pd.read_csv(DATA_DIR / "lane_neighbourhood_join.csv")
parking = pd.read_csv(DATA_DIR / "parking_neighbourhood_join.csv")

# Write to SQLite tables
neighbourhoods.to_sql("Neighbourhood", conn, index=False, if_exists="replace")
accidents.to_sql("CyclingAccident", conn, index=False, if_exists="replace")
lanes.to_sql("BikeLaneSegment", conn, index=False, if_exists="replace")
parking.to_sql("BikeParkingFacility", conn, index=False, if_exists="replace")

print("Tables created:")
cursor = conn.execute("SELECT name FROM sqlite_master WHERE type='table'")
for row in cursor:
    table_name = row[0]
    count = conn.execute(f"SELECT COUNT(*) FROM {table_name}").fetchone()[0]
    print(f"- {table_name}: {count} rows")

Tables created:
- Neighbourhood: 111 rows
- CyclingAccident: 1653 rows
- BikeLaneSegment: 5467 rows
- BikeParkingFacility: 7 rows


**Cypher equivalent:**  
No direct analytical Cypher equivalent. This setup block loads the CSV files into SQLite tables. In Neo4j, the equivalent preparation step is the `LOAD CSV` import process that creates nodes and relationships in the KG.

## 2. Simple query: Accident count per neighbourhood

This is straightforward in both SQL and Cypher.

In [3]:
sql_q1 = """
SELECT
    n.BUURTCODE,
    n.BUURTNAAM AS neighbourhood,
    COUNT(a.accident_id) AS accident_count
FROM Neighbourhood n
LEFT JOIN CyclingAccident a
    ON n.BUURTCODE = a.BUURTCODE
GROUP BY
    n.BUURTCODE,
    n.BUURTNAAM
ORDER BY
    accident_count DESC
LIMIT 10;
"""

result_q1 = pd.read_sql(sql_q1, conn)
display(result_q1)

,BUURTCODE,neighbourhood,accident_count
0,111,Binnenstad,89
1,621,Hurk,39
2,211,Irisbuurt,38
3,321,Doornakkers-West,38
4,425,Rapenland,38
5,412,Hemelrijken,37
6,413,Gildebuurt,37
7,731,Genderbeemd,37
8,522,Achtse Barrier-Gunterslaer,36
9,114,Fellenoord,35


**Cypher equivalent** (run in Neo4j):

```cypher
MATCH (a:CyclingAccident)-[:OCCURRED_IN]->(n:Neighbourhood)
RETURN n.BUURTCODE AS BUURTCODE,
       n.BUURTNAAM AS neighbourhood,
       count(a) AS accident_count
ORDER BY accident_count DESC
LIMIT 10;
```

## 3. Severity breakdown per neighbourhood

Now we add accident severity. In the graph, this uses the `HAS_SEVERITY` relationship to an `AccidentSeverity` node. In SQL, we use a CASE WHEN on the string column.

In [4]:
sql_q2 = """
SELECT
    n.BUURTCODE,
    n.BUURTNAAM AS neighbourhood,
    COUNT(a.accident_id) AS accident_count,
    SUM(CASE WHEN a.accident_outcome_standardized = 'injury' THEN 1 ELSE 0 END) AS injury_count,
    SUM(CASE WHEN a.accident_outcome_standardized = 'fatal' THEN 1 ELSE 0 END) AS fatal_count,
    SUM(CASE WHEN a.accident_outcome_standardized = 'property_damage_only' THEN 1 ELSE 0 END) AS property_damage_only_count
FROM Neighbourhood n
LEFT JOIN CyclingAccident a
    ON n.BUURTCODE = a.BUURTCODE
GROUP BY
    n.BUURTCODE,
    n.BUURTNAAM
ORDER BY
    accident_count DESC
LIMIT 10;
"""

result_q2 = pd.read_sql(sql_q2, conn)
display(result_q2)

,BUURTCODE,neighbourhood,accident_count,injury_count,fatal_count,property_damage_only_count
0,111,Binnenstad,89,18,0,71
1,621,Hurk,39,7,1,31
2,211,Irisbuurt,38,7,0,31
3,321,Doornakkers-West,38,7,1,30
4,425,Rapenland,38,4,0,34
5,412,Hemelrijken,37,10,0,27
6,413,Gildebuurt,37,5,0,32
7,731,Genderbeemd,37,2,0,35
8,522,Achtse Barrier-Gunterslaer,36,9,0,27
9,114,Fellenoord,35,7,0,28


**Cypher equivalent** (run in Neo4j):

```cypher
MATCH (a:CyclingAccident)-[:OCCURRED_IN]->(n:Neighbourhood)
MATCH (a)-[:HAS_SEVERITY]->(s:AccidentSeverity)
RETURN n.BUURTCODE AS BUURTCODE,
       n.BUURTNAAM AS neighbourhood,
       count(a) AS accident_count,
       sum(CASE WHEN s.name = "injury" THEN 1 ELSE 0 END) AS injury_count,
       sum(CASE WHEN s.name = "fatal" THEN 1 ELSE 0 END) AS fatal_count,
       sum(CASE WHEN s.name = "property_damage_only" THEN 1 ELSE 0 END) AS property_damage_only_count
ORDER BY accident_count DESC
LIMIT 10;
```

## 4. Correct cross-dataset SQL baseline: avoiding the fan trap

This is where the structural difference becomes clear.

We want to combine accident counts, lane length, and parking capacity for each neighbourhood. Because `Neighbourhood` has several one-to-many relationships, a naive direct join would create a fan trap and inflate counts.

Therefore, the correct SQL workflow must pre-aggregate accidents, lane lengths, and parking capacity in separate subqueries before joining the summaries back to `Neighbourhood`.

In [5]:
sql_q3 = """
SELECT 
    n.BUURTCODE,
    n.BUURTNAAM AS neighbourhood,
    n.population,
    n.area_km2,
    n.pop_density,

    COALESCE(acc.accident_count, 0) AS accident_count,
    COALESCE(acc.injury_count, 0) AS injury_count,
    COALESCE(acc.fatal_count, 0) AS fatal_count,
    COALESCE(acc.property_damage_only_count, 0) AS property_damage_only_count,

    COALESCE(lane.total_lane_m, 0) AS lane_length_m,

    COALESCE(park.parking_count, 0) AS parking_count,
    COALESCE(park.total_capacity, 0) AS parking_capacity,

    CASE WHEN n.population > 0 
         THEN ROUND(COALESCE(acc.accident_count, 0) * 1000.0 / n.population, 2) 
         ELSE NULL END AS accident_rate_per_1000_residents,

    CASE WHEN n.area_km2 > 0 
         THEN ROUND(COALESCE(lane.total_lane_m, 0) / n.area_km2, 2) 
         ELSE NULL END AS lane_density_m_per_km2,

    CASE WHEN n.population > 0 
         THEN ROUND(COALESCE(park.total_capacity, 0) * 1000.0 / n.population, 2)
         ELSE NULL END AS parking_capacity_per_1000_residents

FROM Neighbourhood n

-- Subquery A: pre-aggregate accidents by neighbourhood
LEFT JOIN (
    SELECT 
        BUURTCODE,
        COUNT(accident_id) AS accident_count,
        SUM(CASE WHEN accident_outcome_standardized = 'injury' THEN 1 ELSE 0 END) AS injury_count,
        SUM(CASE WHEN accident_outcome_standardized = 'fatal' THEN 1 ELSE 0 END) AS fatal_count,
        SUM(CASE WHEN accident_outcome_standardized = 'property_damage_only' THEN 1 ELSE 0 END) AS property_damage_only_count
    FROM CyclingAccident
    WHERE BUURTCODE IS NOT NULL AND BUURTCODE != ''
    GROUP BY BUURTCODE
) acc ON n.BUURTCODE = acc.BUURTCODE

-- Subquery B: pre-aggregate bike lane length by neighbourhood
LEFT JOIN (
    SELECT 
        BUURTCODE,
        SUM(intersected_length_m) AS total_lane_m
    FROM BikeLaneSegment
    WHERE BUURTCODE IS NOT NULL AND BUURTCODE != ''
    GROUP BY BUURTCODE
) lane ON n.BUURTCODE = lane.BUURTCODE

-- Subquery C: pre-aggregate formal bike parking by neighbourhood
LEFT JOIN (
    SELECT 
        BUURTCODE,
        COUNT(facility_id) AS parking_count,
        SUM(COALESCE(capacity, 0)) AS total_capacity
    FROM BikeParkingFacility
    WHERE BUURTCODE IS NOT NULL AND BUURTCODE != ''
    GROUP BY BUURTCODE
) park ON n.BUURTCODE = park.BUURTCODE

ORDER BY 
    accident_rate_per_1000_residents DESC,
    lane_density_m_per_km2 ASC
LIMIT 20;
"""

result_q3 = pd.read_sql(sql_q3, conn)

print("SQL baseline: combined priority candidate query")
display(result_q3)

SQL baseline: combined priority candidate query


,BUURTCODE,neighbourhood,population,area_km2,pop_density,accident_count,injury_count,fatal_count,property_damage_only_count,lane_length_m,parking_count,parking_capacity,accident_rate_per_1000_residents,lane_density_m_per_km2,parking_capacity_per_1000_residents
0,628,Mispelhoef,23.0,2.961939,7.765184,26,10,0,16,11435.162989,0,0.0,1130.43,3860.70,0.00
1,621,Hurk,53.0,2.077475,25.511745,39,7,1,31,6637.077455,0,0.0,735.85,3194.78,0.00
2,545,Esp,17.0,0.932345,18.233601,7,2,0,5,3227.154953,0,0.0,411.76,3461.33,0.00
3,726,Gennep,42.0,1.721451,24.398017,10,0,0,10,4756.936397,0,0.0,238.10,2763.33,0.00
4,637,Flight Forum,63.0,1.381491,45.602898,14,5,0,9,11512.189275,0,0.0,222.22,8333.16,0.00
5,114,Fellenoord,194.0,0.221464,875.989956,35,7,0,28,3176.165818,1,200.0,180.41,14341.70,1030.93
6,636,Park Forum,24.0,0.981788,24.445191,4,3,0,1,2933.532813,0,0.0,166.67,2987.95,0.00
7,226,Sportpark Aalsterweg,32.0,0.565655,56.571590,4,1,0,3,2464.100156,0,0.0,125.00,4356.19,0.00
8,631,BeA2,23.0,2.620186,8.778001,2,1,0,1,7059.812416,0,0.0,86.96,2694.39,0.00
9,632,Meerbos,47.0,1.233203,38.112142,2,1,0,1,5597.158842,0,0.0,42.55,4538.72,0.00


In [6]:
output_path = RESULT_DIR / "sql_baseline_combined_priority_result.csv"

result_q3.to_csv(output_path, index=False)

print(f"Saved SQL baseline result to: {output_path}")

Saved SQL baseline result to: ../data/sql/results/sql_baseline_combined_priority_result.csv


**Cypher equivalent** (run in Neo4j):

```cypher
MATCH (n:Neighbourhood)
WHERE n.population > 0

OPTIONAL MATCH (a:CyclingAccident)-[:OCCURRED_IN]->(n)
OPTIONAL MATCH (a)-[:HAS_SEVERITY]->(s:AccidentSeverity)
WITH n,
     count(a) AS accident_count,
     sum(CASE WHEN s.name = "injury" THEN 1 ELSE 0 END) AS injury_count,
     sum(CASE WHEN s.name = "fatal" THEN 1 ELSE 0 END) AS fatal_count,
     sum(CASE WHEN s.name = "property_damage_only" THEN 1 ELSE 0 END) AS property_damage_only_count

OPTIONAL MATCH (b:BikeLaneSegment)-[r:PASSES_THROUGH]->(n)
WITH n,
     accident_count,
     injury_count,
     fatal_count,
     property_damage_only_count,
     coalesce(sum(r.intersected_length_m), 0) AS lane_length_m

OPTIONAL MATCH (p:BikeParkingFacility)-[:LOCATED_IN]->(n)
WITH n,
     accident_count,
     injury_count,
     fatal_count,
     property_damage_only_count,
     lane_length_m,
     count(p) AS parking_count,
     coalesce(sum(p.capacity), 0) AS parking_capacity

RETURN n.BUURTCODE AS BUURTCODE,
       n.BUURTNAAM AS neighbourhood,
       n.population AS population,
       n.area_km2 AS area_km2,
       n.pop_density AS pop_density,

       accident_count,
       injury_count,
       fatal_count,
       property_damage_only_count,

       lane_length_m,
       parking_count,
       parking_capacity,

       round(accident_count * 1000.0 / n.population, 2) AS accident_rate_per_1000_residents,
       round(lane_length_m / n.area_km2, 2) AS lane_density_m_per_km2,
       round(parking_capacity * 1000.0 / n.population, 2) AS parking_capacity_per_1000_residents

ORDER BY accident_rate_per_1000_residents DESC,
         lane_density_m_per_km2 ASC
LIMIT 20;
```

The SQL version must pre-aggregate accidents, bike lane length, and parking capacity in separate subqueries before joining them back to `Neighbourhood`. The Cypher version follows explicit KG relationships and uses `WITH` to aggregate each relationship pattern step by step.

## 5. Demonstrate the fan trap

To show that the fan trap is a real problem, let's run the **naive** (incorrect) SQL and compare the result with the correct version.

In [7]:
sql_fan_trap = """
SELECT
    n.BUURTNAAM AS neighbourhood,
    COUNT(a.accident_id) AS inflated_accident_count,
    COUNT(l.id) AS inflated_lane_count,
    COUNT(p.facility_id) AS inflated_parking_count
FROM Neighbourhood n
LEFT JOIN CyclingAccident a
    ON n.BUURTCODE = a.BUURTCODE
LEFT JOIN BikeLaneSegment l
    ON n.BUURTCODE = l.BUURTCODE
LEFT JOIN BikeParkingFacility p
    ON n.BUURTCODE = p.BUURTCODE
GROUP BY
    n.BUURTCODE,
    n.BUURTNAAM
ORDER BY
    inflated_accident_count DESC
LIMIT 10;
"""

fan_trap_result = pd.read_sql(sql_fan_trap, conn)
display(fan_trap_result)

,neighbourhood,inflated_accident_count,inflated_lane_count,inflated_parking_count
0,Binnenstad,38270,38270,38270
1,TU-terrein,9112,9112,0
2,Hurk,3237,3237,0
3,Het Ven,3150,3150,0
4,Genderbeemd,3108,3108,0
5,Achtse Barrier-Gunterslaer,3060,3060,0
6,Blixembosch-Oost,2814,2814,0
7,Fellenoord,2660,2660,2660
8,Irisbuurt,2660,2660,0
9,Rapenland,2508,2508,0


In [9]:
# Compare inflated vs correct values
comparison = fan_trap_result[['neighbourhood', 'inflated_accident_count']].merge(
    result_q3[['neighbourhood', 'accident_count']].head(10),
    on='neighbourhood', how='inner'
)
comparison['inflation_factor'] = comparison['inflated_accident_count'] / comparison['accident_count']
print("Fan trap inflation comparison:")
display(comparison)

Fan trap inflation comparison:


,neighbourhood,inflated_accident_count,accident_count,inflation_factor
0,Hurk,3237,39,83.0
1,Fellenoord,2660,35,76.0


This query is intentionally incorrect. It directly joins multiple one-to-many tables to `Neighbourhood`, which creates a fan trap and inflates counts. The correct SQL baseline therefore has to pre-aggregate accidents, lane lengths, and parking capacity separately before joining the summaries back to neighbourhoods.

## 6. SQL vs Cypher comparison

| Aspect | Table-based SQL workflow | KG / Cypher workflow |
|---|---|---|
| Numerical indicators | Can reproduce the same results | Can reproduce the same results |
| Relationship logic | Rebuilt manually through joins on `BUURTCODE` | Explicit in graph relationships |
| Accident severity | Filtered as a column value | Traversed through `HAS_SEVERITY` |
| Lane coverage | Aggregated from flat join table | `SUM` over `PASSES_THROUGH` relationships |
| Missing parking | Requires `LEFT JOIN` and `COALESCE` | Uses `OPTIONAL MATCH` |
| Main value | Good for one-time static analysis | Better semantic integration and extensibility |

## 7. Honest justification


The SQL baseline shows that the same numerical indicators can be reproduced in a table-based workflow. Since the current project uses four static CSV files and the spatial joins were already performed in Python, SQL or pandas can calculate the same accident rates, lane densities, and parking capacity indicators.

However, the knowledge graph provides value as a semantic integration layer. In the table-based workflow, the relationships between accidents, lanes, parking facilities, and neighbourhoods must be manually reconstructed through join keys, grouped subqueries, and null handling. In the KG, these relationships are explicitly represented as `OCCURRED_IN`, `PASSES_THROUGH`, `LOCATED_IN`, and `HAS_SEVERITY`.

Therefore, the advantage of the KG is not that SQL or pandas cannot compute the same numbers, but that the graph makes domain relationships explicit, queryable, and easier to extend with new concepts such as road quality, traffic signals, cycling volume, or demographic changes.

In [8]:
# Clean up
conn.close()
print('SQLite connection closed.')

SQLite connection closed.
